# Chapter 6: The Null Problem: Finding, Filtering, and Filling Missing Values

Companion notebook for *Master Data Cleaning: A Hands-On Beginner's Guide with Python, pandas & NumPy*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 6.1 Understanding Missing Data Conventions (NaN, None, and NaT)

In [1]:
import numpy as np
import pandas as pd

### `np.nan` in numeric columns

In [2]:
sales = pd.read_csv('sales_missing.csv')
sales

,product,quantity,price,discount
0,Widget A,50.0,29.99,0.10
1,Widget B,NaN,34.99,0.15
2,Widget C,75.0,NaN,0.05
3,Widget D,30.0,19.99,NaN


In [3]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   product   4 non-null      str    
 1   quantity  3 non-null      float64
 2   price     3 non-null      float64
 3   discount  3 non-null      float64
dtypes: float64(3), str(1)
memory usage: 292.0 bytes


### Why integer columns become floats

In [4]:
pd.read_csv('sales_missing.csv', dtype={'quantity': 'Int64'})

,product,quantity,price,discount
0,Widget A,50,29.99,0.10
1,Widget B,<NA>,34.99,0.15
2,Widget C,75,NaN,0.05
3,Widget D,30,19.99,NaN


### Missing values in text columns

In [5]:
customers = pd.read_csv('customers.csv')
customers

,customer_id,name,email,phone
0,1001,Aisha Rahman,aisha@example.com,212-555-0142
1,1002,Ben Tan,NaN,312-555-0187
2,1003,Chloe Lim,chloe@example.com,NaN


In [6]:
cities = pd.Series(['New York', None, 'Philadelphia'])
cities

0        New York
1             NaN
2    Philadelphia
dtype: str

In [7]:
cities[1]

nan

In [8]:
pd.Series(['Philadelphia', 42, None])

0    Philadelphia
1              42
2            None
dtype: object

### `NaT` in date columns

In [9]:
appointments = pd.read_csv('appointments.csv',
                           parse_dates=['appointment_date'])
appointments

,patient_id,appointment_date,arrival_time
0,P001,2024-01-15,09:00
1,P002,2024-01-15,NaN
2,P003,NaT,10:00
3,P004,2024-01-16,11:00


### `pd.isna()`

In [10]:
pd.isna(np.nan), pd.isna(None), pd.isna(pd.NaT), pd.isna(pd.NA)

(True, True, True, True)

In [11]:
np.nan == np.nan

False

## 6.2 Identifying Nulls: Using isna() and isnull()

In [12]:
students = pd.read_csv('students.csv')
students

,student_id,name,math,science,english,status
0,S101,Alice,85.5,92.0,88.0,Completed
1,S102,Bob,NaN,88.0,90.0,Completed
2,S103,Charlie,90.0,NaN,87.0,In Progress
3,S104,Diana,78.2,85.5,82.0,NaN
4,S105,Emma,95.0,98.0,94.0,Completed
5,S106,Farid,NaN,NaN,75.0,Failed


### `df.isna()`

In [13]:
students.isna()

,student_id,name,math,science,english,status
0,False,False,False,False,False,False
1,False,False,True,False,False,False
2,False,False,False,True,False,False
3,False,False,False,False,False,True
4,False,False,False,False,False,False
5,False,False,True,True,False,False


In [14]:
students.isna().equals(students.isnull())

True

In [15]:
students['math'].isna()

0    False
1     True
2    False
3    False
4    False
5     True
Name: math, dtype: bool

### `df.notna()`

In [16]:
students.notna()

,student_id,name,math,science,english,status
0,True,True,True,True,True,True
1,True,True,False,True,True,True
2,True,True,True,False,True,True
3,True,True,True,True,True,False
4,True,True,True,True,True,True
5,True,True,False,False,True,True


### `df.isna().sum()`

In [17]:
students.isna().sum()

student_id    0
name          0
math          2
science       2
english       0
status        1
dtype: int64

In [18]:
students.isna().sum().sum()

np.int64(5)

In [19]:
students.isna().sum(axis=1)

0    0
1    1
2    1
3    1
4    0
5    2
dtype: int64

In [20]:
students.isna().mean()

student_id    0.000000
name          0.000000
math          0.333333
science       0.333333
english       0.000000
status        0.166667
dtype: float64

### Filtering rows with a mask

In [21]:
students[students['math'].isna()]

,student_id,name,math,science,english,status
1,S102,Bob,NaN,88.0,90.0,Completed
5,S106,Farid,NaN,NaN,75.0,Failed


In [22]:
students[students['math'].notna()]

,student_id,name,math,science,english,status
0,S101,Alice,85.5,92.0,88.0,Completed
2,S103,Charlie,90.0,NaN,87.0,In Progress
3,S104,Diana,78.2,85.5,82.0,NaN
4,S105,Emma,95.0,98.0,94.0,Completed


In [23]:
students[students['math'].isna() | students['science'].isna()]

,student_id,name,math,science,english,status
1,S102,Bob,NaN,88.0,90.0,Completed
2,S103,Charlie,90.0,NaN,87.0,In Progress
5,S106,Farid,NaN,NaN,75.0,Failed


## 6.3 Filtering Rows and Columns: Using dropna()

### `df.dropna()`

In [24]:
students.dropna()

,student_id,name,math,science,english,status
0,S101,Alice,85.5,92.0,88.0,Completed
4,S105,Emma,95.0,98.0,94.0,Completed


In [25]:
print(students.shape)
print(students.dropna().shape)

(6, 6)
(2, 6)


### Dropping columns with `axis=1`

In [26]:
students.dropna(axis=1)

,student_id,name,english
0,S101,Alice,88.0
1,S102,Bob,90.0
2,S103,Charlie,87.0
3,S104,Diana,82.0
4,S105,Emma,94.0
5,S106,Farid,75.0


### Dropping empty rows with `how='all'`

In [27]:
students_raw = pd.read_csv('students_raw.csv')
students_raw.dropna(how='all')

,student_id,name,math,science,english,status
0,S101,Alice,85.5,92.0,88.0,Completed
1,S102,Bob,NaN,88.0,90.0,Completed
2,S103,Charlie,90.0,NaN,87.0,In Progress
3,S104,Diana,78.2,85.5,82.0,NaN
4,S105,Emma,95.0,98.0,94.0,Completed
5,S106,Farid,NaN,NaN,75.0,Failed
6,S107,NaN,NaN,NaN,NaN,NaN


### Focusing on key columns with `subset`

In [28]:
students.dropna(subset=['math'])

,student_id,name,math,science,english,status
0,S101,Alice,85.5,92.0,88.0,Completed
2,S103,Charlie,90.0,NaN,87.0,In Progress
3,S104,Diana,78.2,85.5,82.0,NaN
4,S105,Emma,95.0,98.0,94.0,Completed


In [29]:
students.dropna(subset=['math', 'science'], how='any')

,student_id,name,math,science,english,status
0,S101,Alice,85.5,92.0,88.0,Completed
3,S104,Diana,78.2,85.5,82.0,NaN
4,S105,Emma,95.0,98.0,94.0,Completed


In [30]:
score_cols = ['math', 'science', 'english']
students_raw.dropna(subset=score_cols, how='all')

,student_id,name,math,science,english,status
0,S101,Alice,85.5,92.0,88.0,Completed
1,S102,Bob,NaN,88.0,90.0,Completed
2,S103,Charlie,90.0,NaN,87.0,In Progress
3,S104,Diana,78.2,85.5,82.0,NaN
4,S105,Emma,95.0,98.0,94.0,Completed
5,S106,Farid,NaN,NaN,75.0,Failed


### Setting a minimum with `thresh`

In [31]:
students.dropna(thresh=5)

,student_id,name,math,science,english,status
0,S101,Alice,85.5,92.0,88.0,Completed
1,S102,Bob,NaN,88.0,90.0,Completed
2,S103,Charlie,90.0,NaN,87.0,In Progress
3,S104,Diana,78.2,85.5,82.0,NaN
4,S105,Emma,95.0,98.0,94.0,Completed


### Saving the result

In [32]:
students_complete = students.dropna(subset=['math', 'science'])
students_complete.shape

(3, 6)

### Choosing a `dropna()` strategy

In [33]:
students_clean = students_raw.dropna(how='all')
students_clean = students_clean.dropna(subset=['student_id', 'name'])
min_values = int(len(students_clean.columns) * 0.7)
students_clean = students_clean.dropna(thresh=min_values)
print(students_raw.shape, '->', students_clean.shape)

(8, 6) -> (6, 6)


## 6.4 Simple Imputation: Using fillna() with Constants

In [34]:
products = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Widget C', 'Widget D'],
    'sales': [100, np.nan, 250, np.nan],
    'region': ['North', None, 'South', 'East'],
})
products

,product,sales,region
0,Widget A,100.0,North
1,Widget B,NaN,NaN
2,Widget C,250.0,South
3,Widget D,NaN,East


### `df.fillna()`

In [35]:
products.fillna(0)

,product,sales,region
0,Widget A,100.0,North
1,Widget B,0.0,0
2,Widget C,250.0,South
3,Widget D,0.0,East


In [36]:
products.fillna(0).dtypes

product        str
sales      float64
region      object
dtype: object

### Filling one column at a time

In [37]:
products_clean = products.copy()
products_clean['sales'] = products_clean['sales'].fillna(0)
products_clean['region'] = products_clean['region'].fillna('Unknown')
products_clean

,product,sales,region
0,Widget A,100.0,North
1,Widget B,0.0,Unknown
2,Widget C,250.0,South
3,Widget D,0.0,East


### Filling several columns with a dictionary

In [38]:
products.fillna({'sales': 0, 'region': 'Unknown'})

,product,sales,region
0,Widget A,100.0,North
1,Widget B,0.0,Unknown
2,Widget C,250.0,South
3,Widget D,0.0,East


## 6.5 Statistical Imputation: Using fillna() with Mean, Median, or Mode

### Filling with the mean

In [39]:
daily_orders = pd.Series([120, 135, np.nan, 128, np.nan, 900])
daily_orders

0    120.0
1    135.0
2      NaN
3    128.0
4      NaN
5    900.0
dtype: float64

In [40]:
daily_orders.fillna(daily_orders.mean())

0    120.00
1    135.00
2    320.75
3    128.00
4    320.75
5    900.00
dtype: float64

### Filling with the median

In [41]:
daily_orders.fillna(daily_orders.median())

0    120.0
1    135.0
2    131.5
3    128.0
4    131.5
5    900.0
dtype: float64

### Filling with the mode

In [42]:
regions = pd.Series(['North', 'South', 'North', None, 'North', None])
regions.mode()

0    North
dtype: str

In [43]:
regions.fillna(regions.mode()[0])

0    North
1    South
2    North
3    North
4    North
5    North
dtype: str

### Different statistics for different columns

In [44]:
stores = pd.DataFrame({
    'sales': [100, np.nan, 250, np.nan],
    'returns': [5, 10, np.nan, 12],
    'region': ['North', None, 'South', 'North'],
})
fill_values = {
    'sales': stores['sales'].mean(),
    'returns': stores['returns'].median(),
    'region': stores['region'].mode()[0],
}
stores.fillna(fill_values)

,sales,returns,region
0,100.0,5.0,North
1,175.0,10.0,North
2,250.0,10.0,South
3,175.0,12.0,North


### Flagging what you filled

In [45]:
stores['sales_imputed'] = stores['sales'].isna()
stores['sales'] = stores['sales'].fillna(stores['sales'].mean())
stores

,sales,returns,region,sales_imputed
0,100.0,5.0,North,False
1,175.0,10.0,NaN,True
2,250.0,NaN,South,False
3,175.0,12.0,North,True


## 6.6 Time-Based Imputation: Using ffill() and bfill()

### `s.ffill()`

In [46]:
dates = pd.date_range('2024-01-01', periods=6, freq='D')
temperature = pd.Series([22.5, np.nan, np.nan, 25.0, 26.5, np.nan],
                        index=dates)
temperature

2024-01-01    22.5
2024-01-02     NaN
2024-01-03     NaN
2024-01-04    25.0
2024-01-05    26.5
2024-01-06     NaN
Freq: D, dtype: float64

In [47]:
temperature.ffill()

2024-01-01    22.5
2024-01-02    22.5
2024-01-03    22.5
2024-01-04    25.0
2024-01-05    26.5
2024-01-06    26.5
Freq: D, dtype: float64

### `s.bfill()`

In [48]:
temperature.bfill()

2024-01-01    22.5
2024-01-02    25.0
2024-01-03    25.0
2024-01-04    25.0
2024-01-05    26.5
2024-01-06     NaN
Freq: D, dtype: float64

In [49]:
temperature.ffill().bfill()

2024-01-01    22.5
2024-01-02    22.5
2024-01-03    22.5
2024-01-04    25.0
2024-01-05    26.5
2024-01-06    26.5
Freq: D, dtype: float64

### Limiting long gaps with `limit`

In [50]:
stock_price = pd.Series([100.0, np.nan, np.nan, np.nan, np.nan, 105.0])
stock_price.ffill(limit=2)

0    100.0
1    100.0
2    100.0
3      NaN
4      NaN
5    105.0
dtype: float64

In [51]:
stock_price.bfill(limit=2)

0    100.0
1      NaN
2      NaN
3    105.0
4    105.0
5    105.0
dtype: float64

### `df.ffill()`

In [52]:
weather = pd.DataFrame({
    'temperature': [22.5, np.nan, np.nan, 25.0],
    'humidity': [65, 68, np.nan, np.nan],
    'pressure': [1013, np.nan, 1015, np.nan],
})
weather.ffill()

,temperature,humidity,pressure
0,22.5,65.0,1013.0
1,22.5,68.0,1013.0
2,22.5,68.0,1015.0
3,25.0,68.0,1015.0


In [53]:
weather['temperature'] = weather['temperature'].ffill(limit=1)
weather

,temperature,humidity,pressure
0,22.5,65.0,1013.0
1,22.5,68.0,NaN
2,NaN,NaN,1015.0
3,25.0,NaN,NaN


### Sort before you fill

In [54]:
readings = pd.DataFrame({
    'date': pd.to_datetime(['2024-03-03', '2024-03-01',
                            '2024-03-04', '2024-03-02']),
    'stock': [40, 50, np.nan, np.nan],
})
readings.ffill()

,date,stock
0,2024-03-03,40.0
1,2024-03-01,50.0
2,2024-03-04,50.0
3,2024-03-02,50.0


In [55]:
readings.sort_values('date').ffill()

,date,stock
1,2024-03-01,50.0
3,2024-03-02,50.0
0,2024-03-03,40.0
2,2024-03-04,40.0
